In [4]:
!pip install -q google-genai==1.66.0 google-auth==2.47.0 pinecone-client pypdf python-docx nltk

In [2]:
!pip install pinecone-client==4.1.2

  Attempting uninstall: pinecone-client
    Found existing installation: pinecone-client 6.0.0
    Uninstalling pinecone-client-6.0.0:
      Successfully uninstalled pinecone-client-6.0.0


In [5]:
import os
import nltk
from google import genai
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader
from docx import Document
from nltk.tokenize import sent_tokenize

nltk.download("punkt")

# ======================
# CONFIGURA TUS KEYS AQUÍ
# ======================
GEMINI_API_KEY = "AQ.Ab8RN6KOHIqwWgDBt_XOTI1hRMxs6WX3iCh4W5v8CGy1EowryQ"
PINECONE_API_KEY = "pcsk_6SdYzt_LqDwzj9Bh4ffGHT2tkxDioQWG9xcZYM15ka3nRycSLgLaTznrB5KZsh7jWRCWH4"

# Cliente Gemini
client = genai.Client(api_key=GEMINI_API_KEY)

# Cliente Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
index_name = "contadores"

# Si el índice NO existe, lo creamos
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=3072,  # embedding de Gemini 1.5
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [10]:
def leer_pdf(path):
    text = ""
    reader = PdfReader(path)
    for page in reader.pages:
        text += page.extract_text()
    return text

def leer_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def leer_docx(path):
    doc = Document(path)
    return "\n".join([p.text for p in doc.paragraphs])

def extraer_texto(path):
    if path.endswith(".pdf"):
        return leer_pdf(path)
    elif path.endswith(".txt"):
        return leer_txt(path)
    elif path.endswith(".docx"):
        return leer_docx(path)
    else:
        return ""

In [11]:
from google.colab import files

uploaded = files.upload()
documentos = {}

for filename in uploaded.keys():
    texto = extraer_texto(filename)
    documentos[filename] = texto
    print(f"Documento cargado: {filename} ({len(texto)} caracteres)")

Saving conciliacion_tlo140107sg2_2026_10.pdf to conciliacion_tlo140107sg2_2026_10 (1).pdf
Documento cargado: conciliacion_tlo140107sg2_2026_10 (1).pdf (5885 caracteres)


In [12]:
import re
import unicodedata

# ==== Limpieza de IDs para Pinecone ====

def limpiar_id(texto):
    """
    Convierte el nombre del archivo en un ID válido para Pinecone.
    Pinecone requiere IDs ASCII, sin acentos ni caracteres especiales.
    """
    texto = unicodedata.normalize("NFKD", texto)
    texto = texto.encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-zA-Z0-9_-]", "_", texto)
    texto = re.sub(r"_+", "_", texto)
    return texto.strip("_")


# ==== Chunker sin dependencias ====

def crear_chunks(texto, max_tokens=350):
    """
    Divide el texto en chunks aproximados usando puntos como separadores.
    Evita dependencias de NLTK y problemas con punkt_tab.
    """
    frases = texto.split(".")
    chunks = []
    actual = ""

    for f in frases:
        f = f.strip()
        if not f:
            continue

        if len(actual) + len(f) + 1 < max_tokens:
            actual += f + ". "
        else:
            if actual.strip():
                chunks.append(actual.strip())
            actual = f + ". "

    if actual.strip():
        chunks.append(actual.strip())

    return chunks


# ===== Configuración =====

namespace = "gemini-docs"

# Modelo de embeddings
MODELO_EMBED = "gemini-embedding-001"


# ===== Indexación de documentos en Pinecone =====

total_insertados = 0
total_errores = 0

for nombre, texto in documentos.items():
    chunks = crear_chunks(texto)

    # Aquí se limpia el nombre del archivo para usarlo como ID válido
    nombre_limpio = limpiar_id(nombre)

    print(f"Procesando {nombre} → {len(chunks)} chunks")
    print(f"ID limpio: {nombre_limpio}")

    vectors_to_upsert = []

    for i, chunk in enumerate(chunks):
        try:
            emb = client.models.embed_content(
                model=MODELO_EMBED,
                contents=chunk
            )

            vector = emb.embeddings[0].values

            vectors_to_upsert.append({
                "id": f"{nombre_limpio}-{i}",
                "values": vector,
                "metadata": {
                    "texto": chunk,
                    "doc": nombre,
                    "chunk": i
                }
            })

            # Subir en bloques de 25
            if len(vectors_to_upsert) >= 25:
                index.upsert(
                    vectors=vectors_to_upsert,
                    namespace=namespace
                )

                total_insertados += len(vectors_to_upsert)
                print(f"✅ Subidos {len(vectors_to_upsert)} vectores")

                vectors_to_upsert = []

        except Exception as e:
            total_errores += 1
            print(f"❌ Error en chunk {i} de {nombre}: {e}")

    # Subir los vectores restantes
    if vectors_to_upsert:
        try:
            index.upsert(
                vectors=vectors_to_upsert,
                namespace=namespace
            )

            total_insertados += len(vectors_to_upsert)
            print(f"✅ Subidos {len(vectors_to_upsert)} vectores restantes")

        except Exception as e:
            total_errores += len(vectors_to_upsert)
            print(f"❌ Error al subir vectores restantes de {nombre}: {e}")


print("====================================")
print("✅ Proceso terminado")
print(f"Vectores insertados: {total_insertados}")
print(f"Errores: {total_errores}")
print("====================================")

Procesando conciliacion_tlo140107sg2_2026_10 (1).pdf → 19 chunks
ID limpio: conciliacion_tlo140107sg2_2026_10_1_pdf
✅ Subidos 19 vectores restantes
✅ Proceso terminado
Vectores insertados: 19
Errores: 0


In [13]:
def consultar_pinecone(pregunta, k=4):
    # Crear embedding de la pregunta usando el MISMO modelo usado para indexar
    emb = client.models.embed_content(
        model=MODELO_EMBED,
        contents=pregunta
    )

    query_vector = emb.embeddings[0].values

    resultados = index.query(
        vector=query_vector,
        top_k=k,
        include_metadata=True,
        namespace=namespace
    )

    # Depuración: verificar si encontró resultados
    if not resultados["matches"]:
        print("\n❌ DEPURACIÓN: No se encontraron documentos relacionados en Pinecone.")
        return ""

    # Extraer el texto de los resultados
    textos = "\n\n".join([
        m["metadata"]["texto"]
        for m in resultados["matches"]
        if "metadata" in m and "texto" in m["metadata"]
    ])

    # Depuración: mostrar parte del contexto recuperado
    print(f"\n✅ DEPURACIÓN: CONTEXTO RECUPERADO ({len(textos)} caracteres):")
    print(textos[:500])
    print("...\n")

    return textos

In [14]:
def preguntar(pregunta):
    # 🔑 PASO CLAVE: Llama a la función que busca en Pinecone
    contexto = consultar_pinecone(pregunta)

    # 🔎 Depuración: Puedes imprimir el contexto para asegurarte de que se recuperó algo.
    # print(f"\nCONTEXTO RECUPERADO:\n{contexto}\n")

    prompt = f"""
    Usa EXCLUSIVAMENTE el siguiente contexto para responder:

    CONTEXTO:
    {contexto}

    PREGUNTA:
    {pregunta}

    INSTRUCCIONES:
    - No inventes información
    - No agregues detalles que no estén en el contexto
    - Responde de forma clara y directa
    """

    # Configuración
    config_params = genai.types.GenerateContentConfig(
        temperature=0.7,
    )

    respuesta = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=config_params
    )

    return respuesta.text

In [18]:
respuesta = preguntar("de que es el documento")
print("🧠 Respuesta de la IA:")
print(respuesta)


✅ DEPURACIÓN: CONTEXTO RECUPERADO (1681 caracteres):
Este comprobante se construye como una referencia de los términos en 
que la operación se realizó, el único comprobante oficial es el estado de 
cuenta que emite Banco Santander S. A.

Posteriormente, el nombre la unidad y 
después el objetivo de la unidad.

Los  requisitos  previamente  señalados  en  el  presente  documento  corresponden  a  la
generalidad  de  los  contribuyentes,  para  casos  específicos,  deberás  consultar  la  ficha  de
trámite 14/CFF Solicitud de generación del Certific
...

🧠 Respuesta de la IA:
Según el contexto proporcionado, el documento es una **representación impresa de un CFDI Versión 4** y sirve como una **referencia de los términos en que se realizó la operación**.
